# Лекция: Регрессионный анализ в Python

**Дисциплина:** Введение в анализ больших данных

Линейная регрессия:

$$
Y' = b_0 + b_1 x_1 + \ldots + b_k x_k
$$

- $Y'$ — предсказанные значения;
- $b_0, b_1, \ldots$ — коэффициенты (МНК);
- остаток $d = Y - Y'$.

**Проверка модели:** F-тест, t-тесты коэффициентов, анализ остатков (нормальность, автокорреляция, гомоскедастичность, MSE).

Инструмент: **`statsmodels`** (`ols`).

Демо на датасете **penguins** (масса тела ~ длина плавника и клюва). Примеры **не совпадают** с лабораторным заданием — выполните его самостоятельно.


## 0. Импорт


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_breuschpagan

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
print("Библиотеки загружены")


---
## 1. Данные и предварительный анализ

Цель: объяснить `body_mass_g` через `flipper_length_mm` и `bill_length_mm`.


In [ ]:
penguins = sns.load_dataset("penguins").dropna()
cols = ["body_mass_g", "flipper_length_mm", "bill_length_mm"]
df = penguins[cols].copy()
print(df.head())
print(df.describe().round(1))


In [ ]:
print("=== Shapiro–Wilk ===")
for c in cols:
    W, p = stats.shapiro(df[c])
    print(f"{c:20s}: W={W:.4f}, p={p:.4g}")

print("\nКорреляции (Spearman):")
print(df.corr(method="spearman").round(3))


In [ ]:
sns.pairplot(df)
plt.suptitle("Парные зависимости", y=1.02)
plt.show()


---
## 2. Множественная линейная регрессия

```text
smf.ols("y ~ x1 + x2", data=df).fit()
```

В `summary()` смотрите:
- **R² / Adj. R²** — доля объяснённой дисперсии;
- **F-statistic** и Prob(F) — значимость модели в целом;
- **coef, t, P>|t|** — значимость отдельных предикторов.


In [ ]:
model = smf.ols(
    "body_mass_g ~ flipper_length_mm + bill_length_mm",
    data=df
).fit()
print(model.summary())


---
## 3. Анализ остатков

1. **Нормальность** — Shapiro–Wilk / Q–Q plot  
2. **Автокорреляция** — Breusch–Godfrey  
3. **Гетероскедастичность** — Breusch–Pagan  
4. **MSE** — средний квадрат ошибки


In [ ]:
resid = model.resid

W, p_sw = stats.shapiro(resid)
print(f"Shapiro–Wilk остатков: W={W:.4f}, p={p_sw:.4g}")
print("  → ок" if p_sw > 0.05 else "  → отклонение от нормальности")

bg = acorr_breusch_godfrey(model, nlags=1)
print(f"\nBreusch–Godfrey: LM={bg[0]:.4f}, p={bg[1]:.4g}")
print("  → нет автокорреляции" if bg[1] > 0.05 else "  → возможна автокорреляция")

bp = het_breuschpagan(resid, model.model.exog)
print(f"\nBreusch–Pagan: BP={bp[0]:.4f}, p={bp[1]:.4g}")
print("  → гомоскедастичность" if bp[1] > 0.05 else "  → гетероскедастичность")

mse = np.mean(resid ** 2)
print(f"\nMSE = {mse:.2f}, RMSE = {np.sqrt(mse):.2f}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

axes[0, 0].scatter(model.fittedvalues, resid, edgecolors="k", alpha=0.6, s=30)
axes[0, 0].axhline(0, color="red", ls="--")
axes[0, 0].set_xlabel("Fitted")
axes[0, 0].set_ylabel("Residuals")
axes[0, 0].set_title("Residuals vs Fitted")

sm.qqplot(resid, line="s", ax=axes[0, 1])
axes[0, 1].set_title("Normal Q-Q")

axes[1, 0].scatter(model.fittedvalues, np.sqrt(np.abs(resid)),
                   edgecolors="k", alpha=0.6, s=30)
axes[1, 0].set_xlabel("Fitted")
axes[1, 0].set_ylabel("sqrt(|Residual|)")
axes[1, 0].set_title("Scale-Location")

axes[1, 1].scatter(df["body_mass_g"], model.fittedvalues,
                   edgecolors="k", alpha=0.6, s=30)
mn = min(df["body_mass_g"].min(), model.fittedvalues.min())
mx = max(df["body_mass_g"].max(), model.fittedvalues.max())
axes[1, 1].plot([mn, mx], [mn, mx], "r--")
axes[1, 1].set_xlabel("Observed")
axes[1, 1].set_ylabel("Predicted")
axes[1, 1].set_title("Observed vs Predicted")

plt.tight_layout()
plt.show()


---
## 4. Простая регрессия (один предиктор)

Для сравнения — только `flipper_length_mm`.


In [ ]:
m_simple = smf.ols("body_mass_g ~ flipper_length_mm", data=df).fit()
print(m_simple.summary().tables[1])
print(f"\nR² simple = {m_simple.rsquared:.3f}")
print(f"R² full   = {model.rsquared:.3f}")


### Как читать результаты (кратко)

1. Высокий **Adj. R²** и малый Prob(F) → модель в целом полезна.  
2. У предиктора **P>|t| < 0.05** → коэффициент статистически отличим от нуля.  
3. Остатки без структуры на графике Residuals vs Fitted, точки около линии на Q–Q → предпосылки ближе к выполнению.  
4. Большой MSE / систематический паттерн в остатках → модель стоит пересмотреть (преобразования, другие предикторы).


---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Модель | `smf.ols("y ~ x1 + x2", data=df).fit()` |
| Сводка | `model.summary()` |
| Остатки / fitted | `model.resid`, `model.fittedvalues` |
| Shapiro остатков | `stats.shapiro(model.resid)` |
| Автокорреляция | `acorr_breusch_godfrey(model, nlags=1)` |
| Гетероскедастичность | `het_breuschpagan(resid, model.model.exog)` |
| MSE | `np.mean(resid**2)` |
| Q–Q plot | `sm.qqplot(resid, line="s")` |

---
## Что сделать после лекции

1. Повторите `ols` на **других** предикторах (например, добавьте `bill_depth_mm`).
2. Откройте лабораторное задание и постройте модель **самостоятельно** на указанных данных.
3. Всегда сочетайте числа из `summary()` с графиками остатков.

Удачи!
